Домашнее задание по итогам собеседования

# Задания
1. Физические виды join (**nested loop/merge/hash**). 

2. Освежить знания по **оконным функциям**

3. Сделать задачу с собеседования без использования подзапросов. 
Условие: Есть таблица `marks` с двумя колонками: `FIO (text)`, `mark (int)`. Задача: Вывести кол-во 2-к у учеников с 10-ю 5-ми

4. Есть таблица `students` `(id int, name text, mark int)`
    `id` - уникальное поле, остальные могут повторяться.  

    На Postgresql реализовать **две хранимых процедуры**:
    1) Процедура получает на вход имя студента (name) и возвращает запись с максимальным id для переданного студента.  
    2) Процедура получает на вход id имя и оценку студента, если такого id нет, то запись с ним добавляется, если есть, то данные для этого id обновляются в таблице.

5. Реализовать **декоратор** на языке Python, который принимает параметр `log_to_file`, и при значении `True` данного параметра вывод записывается в **лог**, иначе в **консоль** (для обоих случаев использовать библиотеку `logging`).  
   
    Декоратор при каждом вызове должен выводить:
   - Имя функции
   - Список аргументов (позиционных и именованных)
   - Измеряет и выводит время выполнения функции
   - Сохраняет общее количество вызовов

6. Написать программу на языке python.  

    Есть **два csv файла**:

    1. `nodes.csv`. Пример заполнения (файл должен содержать от 10 до 15 записей):
        ```csv
        id,name
        1,extract
        2,transform_1
        3,transform_2
        4,load
        ```

    2. `dependencies.csv`. Файл с зависимостями между nodes. Зависимости должны формировать ацикличный граф без изолированных вершин.
        ```csv
        parent_id,child_id
        1,2
        1,3
        2,4
        3,4
        ```

    Необходимо из них сформировать **json-файл** вида:
    ```json
    {
        "extract": {
            "node_id": 1,
            "children": ["transform_1", "transform_2"]
        },
        "transform_1": {
            "node_id": 2,
            "children": ["load"]
        },
        "transform_2": {
            "node_id": 3,
            "children": ["load"]
        },
        "load_2": {
            "node_id": 4,
            "children": []
        },
    ```



_______
# 1. SQL. Физические виды join (merge/hash/nested loop)
Изучила)

_______
# 2. SQL. Оконные функции
Вспомнила)

______
# 3. SQL. Задача с собеседования - marks
**Условие:** Есть таблица `marks` с двумя колонками: `FIO (text)`, `mark (int)`.   
**Задача:** Вывести кол-во 2-к у учеников с 10-ю 5-ми.  
Выполнить задачу без использования подзапросов. 

**Генерируем csv файл** для заполнения таблицы в бд (для проверки решений)

In [ ]:
import csv
import random

# Количество записей
rec_count = 500

# Список имен:
names = [
    "Иван Иванов", "Петр Петров", "Мария Смирнова", "Алексей Кузнецов",
    "Елена Васильева", "Дмитрий Орлов", "Ольга Лебедева", "Сергей Михайлов",
    "Наталья Новикова", "Владимир Соколов"
]

# Список записей:
records = []

# Люди с 10 оценками 5:
for _ in range(len(names)):
    name = random.choice(names)
    for _ in range(10):
        records.append([name, 5])

# Остальные записи:
remaining = rec_count - len(records)
for _ in range(remaining):
    name = random.choice(names)
    mark = random.randint(2, 5)
    records.append([name, mark])

# Перемешиваем записи для случайности:
random.shuffle(records)

# Запись в CSV файл:
with open('marks.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['FIO', 'mark'])
    writer.writerows(records)

print("CSV файл 'marks.csv' успешно создан.")

Два решения **с подзапросом** (то, что я решила на собеседовании)

In [ ]:
WITH fives AS (
	SELECT fio
	FROM marks
	WHERE mark = 5
	GROUP BY fio
	HAVING count(mark) = 10
)
SELECT m.fio, count(mark) AS twos
FROM marks m JOIN fives f ON (m.fio = f.fio)
WHERE mark = 2
GROUP BY m.fio
;

In [ ]:
SELECT fio, count(mark) AS twos
FROM marks
WHERE mark = 2 
    AND fio IN (SELECT fio
                FROM marks
                WHERE mark = 5
                GROUP BY fio
                HAVING count(mark) = 10)
GROUP BY fio
;

Два варианта решения **без подзапроса**.

In [ ]:
SELECT 
	fio,
	sum(CASE WHEN mark = 2 THEN 1 ELSE 0 END) AS twos
FROM marks
GROUP BY fio 
HAVING sum(CASE WHEN mark = 5 THEN 1 ELSE 0 END) = 10;
;

In [ ]:
SELECT 
	fio,
	count(CASE WHEN mark = 2 THEN 1 ELSE NULL END) AS twos
FROM marks
GROUP BY fio 
HAVING count(CASE WHEN mark = 5 THEN 1 ELSE NULL END) = 10;
;

_______
# 4. SQL. Процедуры для таблицы students
Есть таблица `students` `(id int, name text, mark int)`  
`id` - уникальное поле, остальные могут повторяться.  

На Postgresql реализовать **две хранимых процедуры**:
1) Процедура получает на вход имя студента (name) и возвращает запись с максимальным id для переданного студента.  
2) Процедура получает на вход id имя и оценку студента, если такого id нет, то запись с ним добавляется, если есть, то данные для этого id обновляются в таблице.

Генерируем файл students.csv

In [2]:
import csv
import random

rec_count = 500

# Список имен:
names = [
    "Иван Иванов", "Петр Петров", "Мария Смирнова", "Алексей Кузнецов",
    "Елена Васильева", "Дмитрий Орлов", "Ольга Лебедева", "Сергей Михайлов",
    "Наталья Новикова", "Владимир Соколов"
]

# Список записей:
records = []

for i in range(1, rec_count + 1):
    name = random.choice(names)
    mark = random.randint(2, 5)
    records.append([i, name, mark])

with open('students.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['id', 'name', 'mark'])
    writer.writerows(records)

print("CSV файл 'students.csv' успешно создан.")

CSV файл 'students.csv' успешно создан.


### Первая процедура 
(функция на самом деле)

In [ ]:
CREATE OR REPLACE FUNCTION get_student_max_id(_name text)
RETURNS TABLE (id int, name text, mark int) 
AS $$ 
	SELECT *
	FROM students s
	WHERE s.id = (SELECT max(s1.id) FROM students s1 WHERE s1.name = _name);
$$ LANGUAGE sql;


-- Пример вызова функции
SELECT * FROM get_student_max_id('Наталья Новикова');

### Вторая процедура (два варианта)

In [ ]:
-- Вариант 1 - INSERT ... ON CONFLICT DO UPDATE
CREATE OR REPLACE PROCEDURE add_update_record(_id int, _name text, _mark int)
AS $$ 
	INSERT INTO students VALUES (_id, _name, _mark)  -- Пытаемся вставить запись
	ON CONFLICT (id)  -- Если произошел конфликт из-за ограничения на первичный ключ ...
	DO UPDATE SET name = _name, mark = _mark;  -- ... обновляем запись
$$ LANGUAGE sql;

-- Вариант 2 - через проверку IF
CREATE OR REPLACE PROCEDURE add_update_record(_id int, _name text, _mark int)
AS $$ 
BEGIN
	PERFORM * FROM students s WHERE s.id = _id;  -- Ищем запись в таблице с id = _id
	IF NOT FOUND THEN
		INSERT INTO students VALUES (_id, _name, _mark);  -- Если не нашли, то добавляем запись
	ELSE
		UPDATE students SET name = _name, mark = _mark WHERE id = _id;  -- Иначе обновляем запись
	END IF;
END
$$ LANGUAGE plpgsql;


-- Пример вызова процедуры
CALL add_update_record(1232, 'Name', 5);

_______
# 5. Python. Декоратор для логирования
Реализовать **декоратор** на языке Python, который принимает параметр `log_to_file`, и при значении `True` данного параметра вывод записывается в **лог**, иначе в **консоль** (для обоих случаев использовать библиотеку `logging`).  
   
Декоратор при каждом вызове должен выводить:
   - Имя функции
   - Список аргументов (позиционных и именованных)
   - Измеряет и выводит время выполнения функции
   - Сохраняет общее количество вызовов

### Решение 1

In [1]:
import logging
import time
from functools import wraps

# Настраиваем логгеры
logger = logging.getLogger(__name__)  # Основной логгер. Передаем в название логгера - название текущего модуля
logger.setLevel(logging.INFO)  # Устанавливаем уровень логирования на INFO

# Создаем два дополнительных логгера - для логирования в консоль и в файл
file_logger = logger.getChild('file')  # Название логгера будет иметь имя вида __name__.file
console_logger = logger.getChild('console')  # Название логгера будет иметь имя вида __name__.console

# Создаем к логгерам два обработчика - для логирования в консоль и в файл
file_handler = logging.FileHandler("decorator_log.log")
file_handler.setLevel(logging.INFO)
console_handler = logging.StreamHandler()
console_handler.setLevel(logging.INFO)

# Создаем форматировщик
formatter = logging.Formatter("%(name)s %(asctime)s %(levelname)s - %(message)s")

# Собираем все воедино - добавляем форматировщик к обработчикам, а обработчики к логгерам
file_handler.setFormatter(formatter)
console_handler.setFormatter(formatter)
file_logger.addHandler(file_handler)
console_logger.addHandler(console_handler)

In [2]:
# Декоратор
def log_function_info(func, log_to_file):
    @wraps(func)
    def wrapper(*args, **kwargs):
        # Делаем проверку, логировать в файл или в консоль. В зависимости от этого выбираем, какой логгер использовать
        if log_to_file:
            cur_logger = file_logger
        else:
            cur_logger = console_logger

        cur_logger.info(f'Calling function {func.__name__}() with args: {args}, kwargs: {kwargs}')  # Логируем имя функции и переданные аргументы
        wrapper.calls += 1  # Добавляем 1 к количеству вызовов функции
        start_time = time.time()  # Замеряем время выполнения

        result = func(*args, **kwargs)  # Вызываем функцию

        end_time = time.time()
        cur_logger.info(f'Function {func.__name__}() execution time: {end_time - start_time:.4f} seconds')  # Логируем время выполнения
        cur_logger.info(f'Function {func.__name__}() was called {wrapper.calls} times')  # Логируем количество вызовов функции

        return result

    wrapper.calls = 0  # Начальное количество вызовов функции
    return wrapper

Применим наш декоратор

In [3]:
# Определяем функции для дальнейшего декорирования
def add(a, b):
    time.sleep(0.1)
    return a + b

def mult(a, b):
    time.sleep(0.5)
    return a * b

In [4]:
# Декорируем функции
logged_file_add = log_function_info(add, True)
logged_console_add = log_function_info(add, False)

logged_file_mult = log_function_info(mult, True)
logged_console_mult = log_function_info(mult, False)

In [5]:
# Используем задекорированные функции
print('File 1 + 2 =', logged_file_add(1, 2))      # логируем в файл
print('File 3 + 4 =', logged_file_add(3, 4))      # логируем в файл
print('Console 5 + 6 =', logged_console_add(5, 6))   # логируем в консоль
print('Console 7 + 8 =', logged_console_add(7, 8))   # логируем в консоль
print('File 1 * 2 =', logged_file_mult(1, 2))     # логируем в файл
print('File 3 * 4 =', logged_file_mult(3, 4))     # логируем в файл
print('Console 5 * 6 =', logged_console_mult(5, 6))  # логируем в консоль
print('Console 7 * 8 =', logged_console_mult(7, 8))  # логируем в консоль

__main__.console 2025-07-13 21:05:25,684 INFO - Calling function add() with args: (5, 6), kwargs: {}


File 1 + 2 = 3
File 3 + 4 = 7


__main__.console 2025-07-13 21:05:25,785 INFO - Function add() execution time: 0.1009 seconds
__main__.console 2025-07-13 21:05:25,785 INFO - Function add() was called 1 times
__main__.console 2025-07-13 21:05:25,786 INFO - Calling function add() with args: (7, 8), kwargs: {}
__main__.console 2025-07-13 21:05:25,888 INFO - Function add() execution time: 0.1015 seconds
__main__.console 2025-07-13 21:05:25,888 INFO - Function add() was called 2 times


Console 5 + 6 = 11
Console 7 + 8 = 15
File 1 * 2 = 2


__main__.console 2025-07-13 21:05:26,891 INFO - Calling function mult() with args: (5, 6), kwargs: {}


File 3 * 4 = 12


__main__.console 2025-07-13 21:05:27,393 INFO - Function mult() execution time: 0.5009 seconds
__main__.console 2025-07-13 21:05:27,393 INFO - Function mult() was called 1 times
__main__.console 2025-07-13 21:05:27,394 INFO - Calling function mult() with args: (7, 8), kwargs: {}


Console 5 * 6 = 30


__main__.console 2025-07-13 21:05:27,895 INFO - Function mult() execution time: 0.5016 seconds
__main__.console 2025-07-13 21:05:27,895 INFO - Function mult() was called 2 times


Console 7 * 8 = 56


Файл `decorator_log.log` выглядит так:
``` log
__main__.file 2025-07-13 21:05:25,482 INFO - Calling function add() with args: (1, 2), kwargs: {}
__main__.file 2025-07-13 21:05:25,584 INFO - Function add() execution time: 0.1012 seconds
__main__.file 2025-07-13 21:05:25,584 INFO - Function add() was called 1 times
__main__.file 2025-07-13 21:05:25,584 INFO - Calling function add() with args: (3, 4), kwargs: {}
__main__.file 2025-07-13 21:05:25,684 INFO - Function add() execution time: 0.1007 seconds
__main__.file 2025-07-13 21:05:25,684 INFO - Function add() was called 2 times
__main__.file 2025-07-13 21:05:25,888 INFO - Calling function mult() with args: (1, 2), kwargs: {}
__main__.file 2025-07-13 21:05:26,389 INFO - Function mult() execution time: 0.5017 seconds
__main__.file 2025-07-13 21:05:26,389 INFO - Function mult() was called 1 times
__main__.file 2025-07-13 21:05:26,389 INFO - Calling function mult() with args: (3, 4), kwargs: {}
__main__.file 2025-07-13 21:05:26,891 INFO - Function mult() execution time: 0.5016 seconds
__main__.file 2025-07-13 21:05:26,891 INFO - Function mult() was called 2 times
```

В данном случае `счетчик` считает количество вызовов функции из **определенного декоратора**. То есть если мы **одну и ту же** изначальную функцию (например, `add()`) используем внутри **разных декораторов**:
``` python
    logged_file_add = log_function_info(add, True)
    logged_console_add = log_function_info(add, False)
```
то для каждого декоратора **счетчик будет свой**, несмотря на то, что выводиться в лог будет **одно и то же** название изначальной функции (`add()`)

Аналогично можно сделать через **словарь** (в котором будут лежать счетчики для каждой из функций (ключ - сама функция)), но проблема останется той же. Счетчик будет считать отдельное значение для каждого декоратора.

In [ ]:
def log_function_info1(func, log_to_file):
    @wraps(func)
    def wrapper(*args, **kwargs):
        # ...
        calls[func] = calls.get(func, 0) + 1
        logger.info(f'Function {func.__name__}() was called {calls[func]} times')
        # ...

    calls = {}  # количество вызовов функции
    return wrapper

Если это не то поведение счетчика, которое ожидается, то мне приходит в голову лишь решать через глобальные переменные, однако мне не очень нравится этот вариант.

### Решение 2
Аналогичное решение, только с использованием **синтаксического сахара декораторов** будет выглядеть вот так:

In [16]:
def log_function_info2(log_to_file):
    def decorator(func):  # Оборачиваем внутреннюю функцию wrapper еще в одну функцию
        @wraps(func)
        def wrapper(*args, **kwargs):
            if log_to_file:
                cur_logger = file_logger
            else:
                cur_logger = console_logger

            cur_logger.info(f'Calling function {func.__name__}() with args: {args}, kwargs: {kwargs}')
            wrapper.calls += 1
            start_time = time.time()

            result = func(*args, **kwargs)

            end_time = time.time()
            cur_logger.info(f'Function {func.__name__}() execution time: {end_time - start_time} seconds')
            cur_logger.info(f'Function {func.__name__}() was called {wrapper.calls} times')

            return result
        wrapper.calls = 0
        return wrapper
    return decorator

@log_function_info2(True)
def add(a, b):
    return a + b

@log_function_info2(False)
def mult(a, b):
    return a * b

print(add(1, 2))   # логируем в файл
print(add(3, 4))   # логируем в файл
print(mult(1, 2))  # логируем в консоль
print(mult(3, 4))  # логируем в консоль

__main__.console 2025-07-13 21:16:54,507 INFO - Calling function mult() with args: (1, 2), kwargs: {}
__main__.console 2025-07-13 21:16:54,508 INFO - Function mult() execution time: 0.0 seconds
__main__.console 2025-07-13 21:16:54,508 INFO - Function mult() was called 1 times
__main__.console 2025-07-13 21:16:54,509 INFO - Calling function mult() with args: (3, 4), kwargs: {}
__main__.console 2025-07-13 21:16:54,509 INFO - Function mult() execution time: 0.0 seconds
__main__.console 2025-07-13 21:16:54,510 INFO - Function mult() was called 2 times


3
7
2
12


Файл `decorator_log.log` выглядит так:
``` log
__main__.file 2025-07-13 21:16:54,507 INFO - Calling function add() with args: (1, 2), kwargs: {}
__main__.file 2025-07-13 21:16:54,507 INFO - Function add() execution time: 0.0 seconds
__main__.file 2025-07-13 21:16:54,507 INFO - Function add() was called 1 times
__main__.file 2025-07-13 21:16:54,507 INFO - Calling function add() with args: (3, 4), kwargs: {}
__main__.file 2025-07-13 21:16:54,507 INFO - Function add() execution time: 0.0 seconds
__main__.file 2025-07-13 21:16:54,507 INFO - Function add() was called 2 times
```

_______
# 6. Python. Генерация json файла для графа зависимостей

Написать программу на языке python.  

Есть **два csv файла**:

1. `nodes.csv`. Пример заполнения (файл должен содержать от 10 до 15 записей):
    ```csv
    id,name
    1,extract
    2,transform_1
    3,transform_2
    4,load
    ```

2. `dependencies.csv`. Файл с зависимостями между nodes. Зависимости должны формировать ацикличный граф без изолированных вершин.
    ```csv
    parent_id,child_id
    1,2
    1,3
    2,4
    3,4
    ```

Необходимо из них сформировать **json-файл** вида:
```json
{
    "extract": {
        "node_id": 1,
        "children": ["transform_1", "transform_2"]
    },
    "transform_1": {
        "node_id": 2,
        "children": ["load"]
    },
    "transform_2": {
        "node_id": 3,
        "children": ["load"]
    },
    "load_2": {
        "node_id": 4,
        "children": []
    },
```


### Решение
Для начала сгенерируем файлы `nodes.csv` и `dependencies.csv`.

`nodes.csv`:
```csv
id,name
1,extract
2,transform_1
3,transform_2
4,load
5,transform_3
6,transform_4
7,validate
8,aggregate
9,filter
10,enrich
11,normalize
12,cleanse
13,aggregate_2
14,finalize
```

`dependencies.csv`:
```csv
parent_id,child_id
1,2
2,3
3,4
4,5
5,6
6,7
7,8
8,9
9,10
10,11
11,12
12,13
13,14
2,5
3,6
5,7
6,8
7,9
8,10
9,11
10,12
11,13
12,14
```

Теперь выполним задание. Для начала, считаем csv файлы в датафреймы pandas
> Это задание можно решить и без библиотеки pandas, но т.к. в задании не было указано, нужно ли использовать только "чистый" python, я позволила себе такую вольность. Без библиотеки pandas потребуется считывать построчно csv файлы, а дальше в нескольких вложенных циклах перебирать значения и составлять словарь примерно по тому же принципу, которому я следую здесь

In [19]:
import pandas as pd

nodes = pd.read_csv('nodes.csv')
dependencies = pd.read_csv('dependencies.csv')

Сделаем join как в SQL - добавим каждому id из dependencies соответствующее имя из nodes. 

In [68]:
named_dependencies = pd.merge(nodes, dependencies, left_on='id', right_on='parent_id', how='left')[['id', 'child_id', 'name']].rename(columns={'id': 'parent_id', 'name':'parent_name'})
named_dependencies = pd.merge(named_dependencies, nodes, left_on='child_id', right_on='id', how='left')[['parent_id', 'parent_name', 'name']].rename(columns={'name':'child_name'})
named_dependencies

,parent_id,parent_name,child_name
0,1,extract,transform_1
1,2,transform_1,transform_2
2,2,transform_1,transform_3
3,3,transform_2,load
4,3,transform_2,transform_4
5,4,load,transform_3
6,5,transform_3,transform_4
7,5,transform_3,validate
8,6,transform_4,validate
9,6,transform_4,aggregate


Пояснение, что происходит в этих двух длинных длинных строках:    

Первая строка: 
```python
named_dependencies = pd.merge(nodes, dependencies, left_on='id', right_on='parent_id', how='left')[['id', 'child_id', 'name']].rename(columns={'id': 'parent_id', 'name': 'parent_name'})
```
1. ```python
    pd.merge(nodes, dependencies, left_on='id', right_on='parent_id', how='left')
    # Соединяем датафреймы nodes и dependencies по столбцам id и parent_id, с помощью left join (чтобы не потерять информацию о node без детей)
    # Таким образом мы добавили к каждому родителю информацию о его ребенке. 
    # Получили датафрейм из 4 столбцов: 'id', 'name', 'parent_id', 'child_id'
    ```
2. ```python
    [['id', 'child_id', 'name']]
    # Вытаскиваем из этого датафрейма только три столбца (то есть избавляемся от столбца 'parent_id')
    # Почему мы не оставляем столбец 'parent_id' вместо 'id'? Потому что в нем будет значение null для node без детей
    ```
3. ```python
    .rename(columns={'id': 'parent_id', 'name': 'parent_name'})
    # Переименовываем столбец 'id' в 'parent_id' и 'name' в 'parent_name'
    # Получили датафрейм из 3 столбцов: 'parent_id', 'child_id', 'parent_name'
    ```
4. ```python
    named_dependencies = ... 
    # Приравниваем это все к named_dependencies
    ```

Вторая строка: 
```python
named_dependencies = pd.merge(named_dependencies, nodes, left_on='child_id', right_on='id', how='left')[['parent_id', 'parent_name', 'name']].rename(columns={'name': 'child_name'})
```
1. ```python
    pd.merge(named_dependencies, nodes, left_on='child_id', right_on='id', how='left') 
    # Соединяем датафреймы named_dependencies и nodes по столбцам child_id и id с помощью left join (чтобы не потерять информацию о node без детей)
    # Таким образом мы добавили названия к каждому child_id. 
    # Получили датафрейм из 5 столбцов: 'id', 'parent_id', 'child_id', 'parent_name', 'name'
    ```
2. ```python
    [['parent_id', 'parent_name', 'name']]
    # Вытаскиваем из этого датафрейма только четыре столбца (то есть избавляемся от столбца 'id' и 'child_id', они нам больше не нужны)
    ```
3. ```python
    .rename(columns={'name':'child_name'}) 
    # Переименовываем столбец 'name' в 'child_name'
    # Получили датафрейм из 3 столбцов: 'parent_id', 'parent_name', 'child_name'
    ```
4. ```python
    named_dependencies = ... 
    # Приравниваем это все к named_dependencies
    ```

Теперь мы можем легко создать словарь, просто пройдясь последовательно по получившемуся датафрейму

In [ ]:
future_json = {}

for row in named_dependencies.itertuples():
    # Если названия родителя до этого не было в ключах, добавляем сам ключ и ...
    if row.parent_name not in future_json:
        future_json[row.parent_name] = {}
        future_json[row.parent_name]['node_id'] = row.parent_id  # ... добавляем node_id родителя
        future_json[row.parent_name]['children'] = []            # ... и пустой список под детей
    
    # Если название ребенка не null/nan, то добавляем его к списку детей
    if pd.notna(row.child_name):
        future_json[row.parent_name]['children'].append(row.child_name)

future_json  

{'extract': {'node_id': 1, 'children': ['transform_1']},
 'transform_1': {'node_id': 2, 'children': ['transform_2', 'transform_3']},
 'transform_2': {'node_id': 3, 'children': ['load', 'transform_4']},
 'load': {'node_id': 4, 'children': ['transform_3']},
 'transform_3': {'node_id': 5, 'children': ['transform_4', 'validate']},
 'transform_4': {'node_id': 6, 'children': ['validate', 'aggregate']},
 'validate': {'node_id': 7, 'children': ['aggregate', 'filter']},
 'aggregate': {'node_id': 8, 'children': ['filter', 'enrich']},
 'filter': {'node_id': 9, 'children': ['enrich', 'normalize']},
 'enrich': {'node_id': 10, 'children': ['normalize', 'cleanse']},
 'normalize': {'node_id': 11, 'children': ['cleanse', 'aggregate_2']},
 'cleanse': {'node_id': 12, 'children': ['aggregate_2', 'finalize']},
 'aggregate_2': {'node_id': 13, 'children': ['finalize']},
 'finalize': {'node_id': 14, 'children': []}}

Теперь можно со спокойной душой превращать это в json

In [72]:
import json

with open('dependencies.json', 'w') as file:
    json.dump(future_json, file, indent=4)

Получился вот такой json:
```json
{
    "extract": {
        "node_id": 1,
        "children": [
            "transform_1"
        ]
    },
    "transform_1": {
        "node_id": 2,
        "children": [
            "transform_2",
            "transform_3"
        ]
    },
    "transform_2": {
        "node_id": 3,
        "children": [
            "load",
            "transform_4"
        ]
    },
    
    ...

    "aggregate_2": {
        "node_id": 13,
        "children": [
            "finalize"
        ]
    },
    "finalize": {
        "node_id": 14,
        "children": []
    }
}
```